In [0]:
from pyspark.sql.types import (StructType,StructField,StringType,DoubleType,   LongType,TimestampType)
from pyspark.sql.functions import (col,from_json)

In [0]:
bronze_stream_df = (
    spark.readStream
        .format("delta")
        .load(
            "s3a://sebastian-crypto-lakehouse/bronze/crypto_transactions/"
        )
)

In [0]:
trade_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", DoubleType(), True),
    StructField("side", StringType(), True),
    StructField("trade_id", LongType(), True),
    StructField("timestamp", TimestampType(), True)
])

In [0]:
silver_parsed_df = bronze_stream_df.withColumn(
    "parsed_json",
    from_json(col("raw_json"), trade_schema)
    )

In [0]:
silver_df = silver_parsed_df.select(
    col("parsed_json.symbol").alias("symbol"),
    col("parsed_json.price").alias("price"),
    col("parsed_json.quantity").alias("quantity"),
    col("parsed_json.side").alias("side"),
    col("parsed_json.trade_id").alias("trade_id"),
    col("parsed_json.timestamp").alias("event_timestamp"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp").alias("kafka_timestamp")
)

In [0]:
silver_test_query = (
    silver_df.writeStream
        .format("memory")
        .queryName("silver_test")
        .outputMode("append")
        .trigger(availableNow=True)
        .option(
            "checkpointLocation",
            "s3a://sebastian-crypto-lakehouse/checkpoints/silver/test_query/"
        )
        .start()
)

silver_test_query.awaitTermination()

In [0]:
display(spark.sql("SELECT * FROM silver_test"))

In [0]:
silver_df = silver_df.filter(
    col("symbol").isNotNull()
)

In [0]:
silver_df = silver_df.filter(
    col("price") > 0
)

In [0]:
silver_df = silver_df.filter(
    col("quantity") > 0
)

In [0]:
silver_df = silver_df.filter(
    col("side").isin("buy", "sell")
)

In [0]:
silver_df = silver_df.dropDuplicates(
    ["trade_id"]
)

In [0]:
from pyspark.sql.functions import (
    year,
    month,
    dayofmonth,
    hour
)

silver_df = silver_df.withColumn(
    "day",
    dayofmonth(col("event_timestamp"))
).withColumn(
    "hour",
    hour(col("event_timestamp"))
)

In [0]:
silver_query = (
    silver_df.writeStream
        .format("delta")
        .outputMode("append")
        .partitionBy("day", "hour")
        .option(
            "checkpointLocation",
            "s3a://sebastian-crypto-lakehouse/checkpoints/silver/crypto_transactions/"
        )
        .trigger(availableNow=True)
        .start(
            "s3a://sebastian-crypto-lakehouse/silver/crypto_transactions/"
        )
)

In [0]:
silver_query.awaitTermination()